# 04 – Preprocessing, dropout, irregular sampling
Leakage policy: scalers/imputers are fitted on the unlabeled data to be scored (or on the training split in deployment); labels are never used.

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
root = next(p for p in [Path.cwd(), Path.cwd().parent] if (p / "src").exists())
sys.path.insert(0, str(root))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
try:
    from IPython.display import display, Image
except ImportError:
    display = print
PARSED = root / "data" / "processed" / "gas_sensor_drift_parsed.csv"
USE_STANDIN = False   # True ONLY to test the notebook without UCI data (results then are NOT valid)
if USE_STANDIN:
    from src.synthetic import make_standin
    df = make_standin(5000)
    print("!! STAND-IN DATA - not UCI")
else:
    assert PARSED.exists(), "Run notebook 01 (or `python -m src.data_loader`) first"
    df = pd.read_csv(PARSED)
FIG = root / "results" / "figures"; TAB = root / "results" / "tables"
FIG.mkdir(parents=True, exist_ok=True); TAB.mkdir(parents=True, exist_ok=True)
print(df.shape)

In [ ]:
from src.preprocessing import *
cols = feature_columns(df)
print(missing_value_report(df, cols)['missing_cells'], 'missing cells in the original data')
display(variance_analysis(df, cols).head())
hc = high_correlation_pairs(df, cols, 0.95); print(len(hc), 'feature pairs with |r| >= 0.95')
kept = drop_correlated(df, cols, 0.95); print(len(kept), 'features kept after correlation filter')

## Scalers and no-leakage fitting

In [ ]:
cut = int(0.7 * len(df))
tr, te = df[cols].iloc[:cut].to_numpy(), df[cols].iloc[cut:].to_numpy()
for sc in ['standard','robust','minmax']:
    p = Preprocessor('median', sc).fit(tr)       # statistics from TRAIN only
    Zte = p.transform(te)
    print(sc, 'test mean |.| =', np.abs(Zte.mean(0)).mean().round(3), 'test std =', Zte.std(0).mean().round(3))

## Nested dimensionality subsets (16 < 32 < 64 < 128)

In [ ]:
for d in [16,32,64,128]:
    c = select_dimensions(cols, d); print(d, len(c), c[:3], '...')

## Dropout + imputation accuracy (measured on the values we hid)

In [ ]:
from sklearn.preprocessing import StandardScaler
Xs = StandardScaler().fit_transform(df[select_dimensions(cols,64)].iloc[:1500].to_numpy())
rows=[]
for strat in ['random','sensor','consecutive']:
    for frac in [0.05,0.10,0.20,0.30]:
        Xm, mask = simulate_dropout(Xs, select_dimensions(cols,64), frac, strat, seed=0)
        for imp in ['mean','median','knn']:
            Xi = Preprocessor(imp,'none').fit_transform(Xm)
            rows.append(dict(strategy=strat, dropout=frac, imputer=imp, rmse=float(np.sqrt(((Xi-Xs)[mask]**2).mean()))))
        for imp in ['ffill','linear','pchip']:
            Xi = fill_missing_stream(Xm, imp)
            rows.append(dict(strategy=strat, dropout=frac, imputer=imp, rmse=float(np.sqrt(((Xi-Xs)[mask]**2).mean()))))
imp_err = pd.DataFrame(rows); imp_err.to_csv(TAB/'imputation_rmse.csv', index=False)
display(imp_err.pivot_table(index=['strategy','dropout'], columns='imputer', values='rmse').round(3))

## Irregular sampling (SIMULATED: UCI has no raw irregular timestamps)

In [ ]:
keep = simulate_irregular_sampling(len(Xs), 0.3, seed=0)
Xi = Xs.copy(); Xi[~keep] = np.nan
for m in ['ffill','linear','pchip','nearest']:
    print(m, 'RMSE on removed rows:', np.sqrt(((fill_missing_stream(Xi, m)-Xs)[~keep]**2).mean()).round(3))

### Result log (fill in after running)
*Not yet executed.* Copy the numbers printed above into your report; do not quote numbers from memory.